## 4.3 CartPole

**(a)**

In [ ]:
def build_network(self) -> torch.nn.Module:
    Q=torch.nn.Sequential(
        torch.nn.Linear(self.state_dim, self.hidden_dim),
        torch.nn.ReLU(),
        torch.nn.Linear(self.hidden_dim, self.hidden_dim),
        torch.nn.ReLU(),
        torch.nn.Linear(self.hidden_dim, self.action_dim)
    )
    return Q

In [ ]:
def policy(self, state : Union[np.ndarray, torch.tensor], train : bool=False) -> torch.Tensor:
    if isinstance(state, np.ndarray):
        state = torch.tensor(state).unsqueeze(0).to(self.device)
    if train:
        if random.random()<self.eps_threshold():
            return torch.randint(0, self.action_dim, (1,)).to(self.device)
        else:
            with torch.no_grad():
                return torch.argmax(self.policy_network(state)).unsqueeze(0)
    else:
        with torch.no_grad():
            return torch.argmax(self.policy_network(state)).unsqueeze(0)

In [ ]:
def train(self, env : gym.wrappers, num_episodes : int=100) -> None:
    optimizer=torch.optim.AdamW(self.policy_network.parameters(), lr=self.learning_rate)
    loss_fn=torch.nn.MSELoss()
    for episode in tqdm(range(num_episodes)):
        state, _ = env.reset()
        done = False
        while not done:
            action=self.policy(state, train=True)
            next_state, reward, terminated, truncated, _ = env.step(action.item())
            done=terminated or truncated
            sp = None if terminated else torch.tensor(next_state).unsqueeze(0).to(self.device)
            self.buffer.append(Transition(torch.tensor(state).unsqueeze(0).to(self.device), action, torch.tensor(reward).unsqueeze(0).to(self.device), sp))
            state=next_state

            if len(self.buffer)>self.batch_size:
                states, actions, targets=self.sample_buffer()
                predictions=self.policy_network(states).gather(1, actions)
                loss=loss_fn(predictions, targets)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_value_(self.policy_network.parameters(), 100)
                optimizer.step()

DQN evaluation on CartPole-v1:
- Before training: average reward = 11.4
- After training (500 episodes): average reward = 500.0

**(b)**

In [ ]:
def learn(self, rewards: list, log_probs: list) -> None:
    T = len(rewards)
    log_probs = torch.stack(log_probs)

    # 1) Naive REINFORCE
    # R = sum(self.gamma**t * rewards[t] for t in range(T))
    # loss = -(log_probs * R).sum()

    # 2) REINFORCE with causality trick
    # returns = torch.zeros(T)
    # for t in range(T):
    #     returns[t] = sum(self.gamma**(tp - t) * rewards[tp] for tp in range(t, T))
    # loss = -(log_probs * returns).sum()

    # 3) REINFORCE with causality trick and baseline to "center" the returns
    returns = torch.zeros(T)
    for t in range(T):
        returns[t] = sum(self.gamma**(tp - t) * rewards[tp] for tp in range(t, T))
    baseline = sum(self.gamma**t * rewards[t] for t in range(T)) / T
    loss = -(log_probs * (returns - baseline)).sum()

In [ ]:
from IPython.display import display, Image

print("Naive REINFORCE:")
display(Image(filename="../reward_curve_naive.png"))
print("REINFORCE with causality trick:")
display(Image(filename="../reward_curve_causality.png"))
print("REINFORCE with causality trick + baseline:")
display(Image(filename="../reward_curve_baseline.png"))

DQN is more sample efficient than REINFORCE because it stores transitions in a replay buffer and performs a gradient step at every environment step, reusing past experience multiple times. REINFORCE, on the other hand, collects trajectories online with the current policy and discards them after a single gradient update, making it inherently less data efficient. As a result, DQN reaches an average reward of 500.0 after 500 episodes while all three REINFORCE variants achieve lower rewards.

Among the REINFORCE variants, the causality trick improves learning since the variance from past rewards is removed from the gradient computation, allowing each action to be reinforced only based on its future consequences. The baseline variant further removes state-relative variance for a similar final performance as the causality trick.